`r format(Sys.time(), '🗓️ %d %B, %Y 🕔 %H:%M')`

# General Stats

In [ ]:
library(BioCircos)
library(dplyr)
library(DT)
library(tidyr)
library(tools)
library(viridisLite)

In [ ]:
readvariants <- function(x){
  y <- read.table(
    x,
    header = T,
    colClasses = c("factor", "integer", "factor", "integer", "integer", "integer", "character", "character", "numeric", "factor", "factor")
  )
  if (nrow(y) == 0){
    return(y)
  }
  y$Population <- gsub(".bedpe", "", basename(x))
  y$Population <- as.factor(y$Population)
  y <- y[, c(12,1:11)]
  return(y)
}

cleanbedpe <- function(x){
  y <- x[x$Chr1 == x$Chr2, c(-4, -11)]
  names(y)[2] <- "Chr1"
  y$Score <- round(y$Score, digits = 3)
  y$Length <- abs(y$Break2 - y$Break1)
  return(y)
}

chimeric <- function(x){
  y <- x[x$Chr1!=x$Chr2, -11]
  y$Score <- round(y$Score, digits = 3)
  return(y)
}

In [ ]:
indir <- params$bedpedir
#indir <- "SV/naibr/bedpe" 
#fai <- "Genome/genome.fai"
fai <- params$faidx
infiles <- list.files(path = indir, pattern = "*bedpe", full.names = T)

In [ ]:
emptyfiles <- character()
sv <- data.frame(matrix(ncol=12,nrow=0, dimnames=list(NULL, c("Population", "Chr1", "Break1", "Chr2", "Break2", "SplitMolecules", "DiscordantReads", "Orientation", "Haplotype", "Score", "PassFilter", "SV"))))
for(i in infiles){
  .df <- readvariants(i)
  if(nrow(.df)==0){
      emptyfiles <- c(emptyfiles, gsub(".bedpe", "", basename(i)))
  } else {
    sv <- rbind(sv, .df)
  }
}
if(length(emptyfiles) == length(infiles)){
  cat("There are no variants detected in any of the populations:\n")
  cat(paste0("- ", emptyfiles), sep = "\n")
  knitr::knit_exit()
}

In [ ]:
sv_clean <- cleanbedpe(sv)
summinfo <- sv %>%
  group_by(Population, SV) %>%
  summarise(tot = n()) %>%
  group_by(SV) %>%
  summarise(count = sum(tot), avg = round(mean(tot, na.rm = T),2 ), sd = round(sd(tot, na.rm = T), 2))

##

In [ ]:
list(
  color = "light",
  value = prettyNum(length(levels(sv$population)), big.mark = ",")
)

In [ ]:
variant <- "deletion"
ndel <- ifelse(variant %in% summinfo$SV, summinfo$avg[which(summinfo$SV == variant)], 0)
list(
  color = "#dfdfdf",
  value =prettyNum(round(ndel,1), big.mark = ",")
)

In [ ]:
variant <- "duplication"
ndup <- ifelse(variant %in% summinfo$SV, summinfo$avg[which(summinfo$SV == variant)], 0)
list(
  color = "#dfdfdf",
  value= prettyNum(round(ndup,1), big.mark = ",")
)

In [ ]:
variant <- "inversion"
ninv <- ifelse(variant %in% summinfo$SV, summinfo$avg[which(summinfo$SV == variant)], 0)
list(
  color = "#dfdfdf",
  value = prettyNum(round(ninv,1), big.mark = ",")
)

In [ ]:
list(
  color = ifelse(length(emptyfiles) == 0, "#dfdfdf", "info"),
  value = length(emptyfiles)
)

##
This report details the structural variants identified by [NAIBR](https://github.com/pontushojer/NAIBR)
[(publication)](https://doi.org/10.1093/bioinformatics/btx712). This tab (`General Stats`) shows overview
information. The `Per-Contig Plot` tab shows you an interactive
plot detailing all variants identified. The variants are given by:

Inversion
: A segment that broke off and reattached within the same chromosome, but in reverse orientation

Duplication
: A type of mutation that involves the production of one or more copies of a gene or region of a chromosome

Deletion
: A type of mutation that involves the loss of one or more nucleotides from a segment of DNA

##
::: {.card title="Variants by Count" expandable="true"}
This table details counts each type of structural variant for every contig for every population.

In [ ]:
grpcounts <- sv_clean %>%
  group_by(Population, Chr1, SV) %>%
  summarise(count = n()) %>%
  pivot_wider(names_from = Population, values_from = count)

DT::datatable(
  grpcounts,
  rownames = F,
  filter = "top",
  extensions = 'Buttons',
  fillContainer = T,
  options = list(
    dom = 'Brtp',
    buttons = list(list(extend = "csv",filename = "naibr_sv_count_pop")),
    scrollX = TRUE
  )
)

:::

### by length 
::: {.card title="Variants by Length" expandable="true"}
This table details the lengths (in bp) of structural variants for every contig for every population.

In [ ]:
grplens <- sv_clean %>%
  group_by(Population, Chr1, SV) %>%
  summarise(bp = sum(Length)) %>%
  pivot_wider(names_from = Population, values_from = bp)

DT::datatable(
  grplens,
  rownames = F,
  filter = "top",
  extensions = 'Buttons',
  fillContainer = T,
  options = list(
    dom = 'Brtp',
    buttons = list(list(extend = "csv",filename = "naibr_sv_count_bp")),
    scrollX = TRUE
  )
)

:::


##
::: {.card title="All Non-Chimeric Variants" expandable="true"}
This table details the variants detected by NAIBR that appear on a single
contig/chromosome and passed the programs internal filtering.

In [ ]:
DT::datatable(
  sv_clean,
  rownames = F,
  filter = "top",
  extensions = 'Buttons', 
  fillContainer = T,
  options = list(
    dom = 'Brtp',
    buttons = list(list(extend = "csv",filename = "naibr_sv")),
    scrollX = TRUE
  )
)

:::

###
::: {.card title="Chimeric Variants" expandable="true"}
This table shows any structural variants whose breakpoints span multiple contigs
and may require further assessment. These variants omitted from the plots.

In [ ]:
DT::datatable(
  chimeric(sv),
  rownames = F,
  filter = "top",
  extensions = 'Buttons',
  fillContainer = T,
  options = list(
    dom = 'Brtp',
    buttons = list(list(extend = "csv",filename = "naibr_sv_chimeric")),
    scrollX = TRUE
  )
)

:::


# Inversions

In [ ]:
fa.sizes <- read.table(fai, header = F) %>% arrange(desc(2))
colnames(fa.sizes) <- c("contig", "size")
plot_contigs <- params$contigs
if (all(plot_contigs == "default")){
  # make sure that the contigs with SV are the ones being plotted, not the others
  fa.sizes <- fa.sizes[fa.sizes$contig %in% unique(sv$Chr1), 1:2]
  # limit the data to only the 30 of the largest present contigs
  fa.sizes <- fa.sizes[1:min(nrow(fa.sizes), 30), ]
} else {
  fa.sizes <- fa.sizes[fa.sizes$contig %in% plot_contigs, 1:2]
}

# filter out variants that aren't on one of these contigs
sv_clean <- sv_clean[sv_clean$Chr1 %in% fa.sizes$contig, ]
sv_clean$Chr1 <- factor(sv_clean$Chr1)

populations <- levels(sv_clean$Population)
color.palette <- turbo(length(populations))
names(color.palette) <- populations

genomeChr <- fa.sizes$size
names(genomeChr) <- fa.sizes$contig
genomeChr <- as.list(genomeChr)
radius <- 0.9
radius_increments <- round(.6 / length(populations), 3)
padding <- radius_increments * 0.2

In [ ]:
calculate_luminance <- function(hex) {
  rgb_values <- col2rgb(hex) / 255
  luminance <- 0.2126 * rgb_values[1] + 0.7152 * rgb_values[2] + 0.0722 * rgb_values[3]
  return(luminance)
}

# Function to decide text color based on background luminance
get_text_color <- function(background_color) {
  luminance <- calculate_luminance(background_color)
  if (luminance > 0.5) {
    # Dark background -> Light text (black)
    return("black") 
  } else {
    # Light background -> Dark text (white)
    return("white") 
  }
}
poplegend <- DT::datatable(
  data.frame(Group = populations),
  rownames = F,
  fillContainer = T,
  options = list(
    dom = 't',
    scrollX = F,
    scrollY = F,
    paging = F
  ),
) %>% formatStyle(
  "Group",
  backgroundColor = styleEqual(names(color.palette), color.palette),
  # Adjust text color dynamically
  color = styleEqual(names(color.palette), sapply(color.palette, get_text_color))
)

## {.fit} 
<h2>Detected Inversions</h2>

This is a circular plot to visualize the distribution of _putative_ inversions across (up to) 30 of the largest contigs
(or the ones that were specified) across the groups you specified in your data.
If you are unfamiliar with this kind of visualization, it's a circular representation of a linear genome.
Each "wedge" is a different contig, from position 0 to  the end of the contig, and is labelled by the contig name.
Each internal (grey) ring is detected inversions on that contig for a different population.
Very small variants may be difficult to see or will not appear on the plot.

##
::: {.card width="15%" title="Legend"}

In [ ]:
poplegend

:::

###

In [ ]:
radius <- 0.9
tracks <- BioCircosTracklist()
variant_type <- "inversion"
# Add one track for each population
for( pop in populations ){
  variant <- sv_clean[(sv_clean$SV == variant_type) & (sv_clean$Population == pop),]
  if(nrow(variant) > 0){
    tracks <- tracks + BioCircosArcTrack(
      paste0(variant_type, pop),
      as.character(variant$Chr1),
      variant$Break1,
      variant$Break2,
      opacities = 0.7,
      values = variant$SplitMolecules,
      minRadius = radius - (radius_increments - padding),
      maxRadius = radius - padding,
      colors = color.palette[pop],
      labels = paste0("Population: ", pop, "</br>Type: Inversion"),
    )
  }
  # loop to create backgrounds
  tracks <- tracks + BioCircosBackgroundTrack(
    paste0(pop,"_background"),
    minRadius = radius - radius_increments, maxRadius = radius,
    fillColors = "#F9F9F9",
    borderColors = "#C9C9C9",
    borderSize = 1
  )
  radius <- radius - radius_increments
}
BioCircos(
  tracks,
  displayGenomeBorder = F,
  genome = genomeChr,
  chrPad = 0.03,
  genomeTicksDisplay = F,
  genomeLabelDy = 3,
  genomeLabelTextSize = 20,
  SNPMouseOverCircleSize = 2,
  SNPMouseOverTooltipsHtml03 = "<br/>Length: ",
  width = "100%",
  height = "1000px",
  elementId = paste0(variant_type, "_circosplot") 
)

# Duplications
## {.fit} 
<h2>Detected Duplications</h2>

This is a circular plot to visualize the distribution of _putative_ duplications across (up to) 30 of the largest contigs
(or the ones that were specified) across the groups you specified in your data.
If you are unfamiliar with this kind of visualization, it's a circular representation of a linear genome.
Each "wedge" is a different contig, from position 0 to  the end of the contig, and is labelled by the contig name.
Each internal (grey) ring is detected inversions on that contig for a different population.
Very small variants may be difficult to see or will not appear on the plot.

##
::: {.card width="15%" title="Legend"}

In [ ]:
poplegend

:::
### Duplications

In [ ]:
# Add one track for each population
radius <- 0.9
tracks <- BioCircosTracklist()
variant_type <- "duplication"
# Add one track for each population
for( pop in populations ){
  variant <- sv_clean[(sv_clean$SV == variant_type) & (sv_clean$Population == pop),]
  if(nrow(variant) > 0){
    tracks <- tracks + BioCircosArcTrack(
      paste0(variant_type, pop),
      as.character(variant$Chr1),
      variant$Break1,
      variant$Break2,
      opacities = 0.7,
      values = variant$SplitMolecules,
      minRadius = radius - (radius_increments - padding),
      maxRadius = radius - padding,
      colors = color.palette[pop],
      labels = paste0("Population: ", pop, "</br>Type: Duplication"),
    )
  }
  # loop to create backgrounds
  tracks <- tracks + BioCircosBackgroundTrack(
    paste0(pop,"_background"),
    minRadius = radius - radius_increments, maxRadius = radius,
    fillColors = "#F9F9F9",
    borderColors = "#C9C9C9",
    borderSize = 1
  )
  radius <- radius - radius_increments
}

BioCircos(
  tracks,
  displayGenomeBorder = F,
  genome = genomeChr,
  chrPad = 0.03,
  genomeTicksDisplay = F,
  genomeLabelDy = 3,
  genomeLabelTextSize = 20,
  width = "100%",
  height = "1000px",
  elementId = paste0(variant_type, "_circosplot") 
)

# Deletions
## {.fit} 
<h2>Detected Deletions</h2>

This is a circular plot to visualize the distribution of _putative_ deletions across (up to) 30 of the largest contigs
(or the ones that were specified) across the groups you specified in your data.
If you are unfamiliar with this kind of visualization, it's a circular representation of a linear genome.
Each "wedge" is a different contig, from position 0 to  the end of the contig, and is labelled by the contig name.
Each internal (grey) ring is detected inversions on that contig for a different population. Very small variants may
be difficult to see or will not appear on the plot. Deletions are displayed as points as they tend to be small
and clump together.

##
::: {.card width="15%" title="Legend"}

In [ ]:
poplegend

:::

In [ ]:
# Add one track for each population
radius <- 0.9
tracks <- BioCircosTracklist()
variant_type <- "deletion"
# Add one track for each population
for( pop in populations ){
  variant <- sv_clean[(sv_clean$SV == variant_type) & (sv_clean$Population == pop),]
  if(nrow(variant) > 0){
    tracks <- tracks + BioCircosSNPTrack(
      paste0(variant_type, pop),
      as.character(variant$Chr1),
      variant$Break1,
      values = abs(variant$Break2 - variant$Break1),
      minRadius = radius - (radius_increments - padding),
      maxRadius = radius - padding,
      size = 2,
      shape = "rect",
      opacities = 0.7,
      colors = color.palette[pop],
      labels = paste0("Population: ", pop, "</br>Type: Deletion"),
  )
  }
  # loop to create backgrounds
  tracks <- tracks + BioCircosBackgroundTrack(
    paste0(pop,"_background"),
    minRadius = radius - radius_increments, maxRadius = radius,
    fillColors = "#F9F9F9",
    borderColors = "#C9C9C9",
    borderSize = 1
  )
  radius <- radius - radius_increments
}

BioCircos(
  tracks,
  displayGenomeBorder = F,
  genome = genomeChr,
  chrPad = 0.03,
  genomeTicksDisplay = F,
  genomeLabelDy = 3,
  genomeLabelTextSize = 20,
  SNPMouseOverCircleSize = 2,
  SNPMouseOverTooltipsHtml03 = "<br/>Length: ",
  width = "100%",
  height = "1000px",
  elementId = paste0(variant_type, "_circosplot") 
)

# Supporting Info
## interpreting output 
::: {.card title="Interpreting NAIBR Output"}

NAIBR outputs a tab-delimited file with named columns (along with a VCF
and a reformatted bedpe file with extra information). The columns of the
bedpe file are deciphered as such:

In [ ]:
knitr::kable(
  data.frame(
    "Column" = names(sv),
    "Descrption" = c(
      "Name of the population",
      "Name of the contig where the variant starts",
      "Base-pair position in `Chr1` of the start of the variant",
      "Name of the contig where the variant ends",
      "Base-pair position in `Chr2` of the end of the variant",
      "Number of split molecules supporting variant",
      "Number of discordant reads supporting variant",
      "Orientation of variant relative to reference",
      "The haplotype of the variant",
      "log-likelihood score for variant",
      "`PASS` if passes internal NAIBR filter threshold, otherwise `FAIL`",
      "The type of variant (inversion, deletion, or duplication)"
    )
  ), col.names = c("Column Name", "Description")
)

:::